# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
ECOHOME_SYSTEM_PROMPT = """You are EcoHome's Energy Advisor, an expert in
residential electricity use, solar generation, electric vehicles, HVAC systems, smart
appliances, and time-of-use pricing. Give practical, personalized recommendations that
reduce costs and environmental impact without compromising safety or essential comfort.

Use the available tools whenever current, historical, or calculated information is needed:
- Check weather before making solar-generation recommendations.
- Check electricity prices before recommending when to run flexible loads.
- Query usage and generation data before describing historical patterns.
- Search the energy knowledge base for relevant best practices.
- Use the savings calculator for numerical savings claims.

Base recommendations on the user's question and supplied household context, including
location, schedule, devices, comfort preferences, solar capacity, battery state, and
charging requirements. Never invent missing measurements or household details. State
assumptions and ask for information when its absence would materially change the answer.

Lead with a clear recommendation and specific time window when applicable. Explain how
pricing, weather, solar output, and usage history support it. Quantify savings when enough
data is available, identify uncertainty and safety constraints, and keep answers concise
and actionable.

Example questions you should handle:
- When should I charge my EV to use the cheapest and cleanest electricity?
- What thermostat schedule balances comfort with peak-rate savings?
- Which appliances should I run during tomorrow's strongest solar-production window?
- How much could I save annually by reducing my HVAC electricity consumption?
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow in San Francisco, I recommend charging between **10 AM and 2 PM**. 

### Here's why:

1. **Solar Generation**: 
   - Solar irradiance is expected to peak around **12 PM** with a value of **850.0 W/m²**, which indicates strong solar production. 
   - The solar output will be significant from **9 AM** (601.0 W/m²) to **2 PM** (with a gradual decrease after 12 PM).

2. **Electricity Pricing**:
   - The electricity rates during the day (from **6 AM to 7 PM**) are at the **peak rate of $0.241 per kWh**.
   - However, the rates drop to **$0.161 per kWh** during the **off-peak hours** from **10 PM to 6 AM** and again after **8 PM**.

### Summary:
- **Charge your EV between 10 AM and 2 PM** to take advantage of solar power, even though the cost is higher during this time. If you can wait until after **8 PM**, you can charge at the lower rate of **$0.161 per kWh**.

If you have any specific charging req

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [7]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "I need my EV charged by 7 AM. Should I charge overnight or wait for daytime solar generation?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should compare overnight prices with forecast solar availability and recommend a charging window",
    },
    {
        "id": "thermostat_1",
        "question": "What thermostat schedule should I use tomorrow to stay comfortable while avoiding peak electricity prices?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should recommend temperature settings and pre-heating or pre-cooling times based on weather and peak rates",
    },
    {
        "id": "thermostat_2",
        "question": "Review my recent HVAC energy use and suggest how I can reduce it without making the house uncomfortable.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should use historical HVAC consumption and provide practical comfort-aware efficiency recommendations",
    },
    {
        "id": "appliance_scheduling_1",
        "question": "When is the cheapest time tomorrow to run my dishwasher and washing machine?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should identify specific off-peak operating windows for both appliances",
    },
    {
        "id": "appliance_scheduling_2",
        "question": "How should I schedule flexible household appliances to use more of my solar power tomorrow?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should align appliance operation with forecast solar production while considering electricity rates",
    },
    {
        "id": "solar_maximization_1",
        "question": "At what hours tomorrow should my solar panels generate the most energy, and which loads should I move into that period?",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should identify the forecast high-generation period and recommend suitable loads to schedule then",
    },
    {
        "id": "solar_maximization_2",
        "question": "Compare my recent solar generation with my household consumption and suggest how to increase self-consumption.",
        "expected_tools": ["query_solar_generation", "query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should compare generation and consumption patterns and recommend ways to use more solar energy on site",
    },
    {
        "id": "cost_savings_1",
        "question": "My EV currently uses 18 kWh per day. If optimization reduces that to 14 kWh at $0.20 per kWh, how much would I save daily and annually?",
        "expected_tools": ["calculate_energy_savings"],
        "expected_response": "The response should calculate the daily energy savings, daily cost savings, percentage reduction, and annual savings",
    },
    {
        "id": "cost_savings_2",
        "question": "Estimate how much I could save by reducing my HVAC usage from 30 kWh to 24 kWh per day using today's electricity price.",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should use current pricing to estimate daily and annual HVAC savings and state any pricing assumption",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [8]:
CONTEXT = "Location: San Francisco, CA"

In [9]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: I need my EV charged by 7 AM. Should I charge overnight or wait for daytime solar generation?
--------------------------------------------------

Test 3: thermostat_1
Question: What thermostat schedule should I use tomorrow to stay comfortable while avoiding peak electricity prices?
--------------------------------------------------

Test 4: thermostat_2
Question: Review my recent HVAC energy use and suggest how I can reduce it without making the house uncomfortable.
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: When is the cheapest time tomorrow to run my dishwasher and washing machine?
--------------------------------------------------

Test 6: appliance_scheduling_2
Question: How should I schedule flexible 

In [10]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Customer context for this request:\nLocation: San Francisco, CA\nUse these details to personalize the recommendation. Do not infer details that are not provided.', additional_kwargs={}, response_metadata={}, id='63bc8986-5d9d-4112-b762-235a2c6c7e8e'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='f957f834-696b-478b-bfeb-ac47905844f9'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1420, 'total_tokens': 1481, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cac

## 4. Evaluate Responses

In [15]:
import os
from statistics import mean
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.metrics import NumericMetric

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is required for Ragas evaluation.")

ragas_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ragas_llm = llm_factory("gpt-4o-mini", client=ragas_client)

RESPONSE_METRICS = {
    "accuracy": NumericMetric(
        name="accuracy",
        allowed_values=(0.0, 1.0),
        prompt="""Score the factual and numerical accuracy of the Energy Advisor response from 0 to 1. Check that claims are internally consistent, do not invent unsupported facts, and satisfy the expected outcome. Question: {question} Expected outcome: {expected_response} Response: {response}""",
    ),
    "relevance": NumericMetric(
        name="relevance",
        allowed_values=(0.0, 1.0),
        prompt="""Score how directly the response addresses the user's energy question from 0 to 1. Penalize tangents and failure to answer the requested decision. Question: {question} Expected outcome: {expected_response} Response: {response}""",
    ),
    "completeness": NumericMetric(
        name="completeness",
        allowed_values=(0.0, 1.0),
        prompt="""Score completeness from 0 to 1 by checking whether the response covers every material requirement in the expected outcome, including analysis, constraints, and requested calculations. Question: {question} Expected outcome: {expected_response} Response: {response}""",
    ),
    "usefulness": NumericMetric(
        name="usefulness",
        allowed_values=(0.0, 1.0),
        prompt="""Score practical usefulness from 0 to 1. A strong response gives clear, specific, safe, and actionable recommendations and states important assumptions. Question: {question} Expected outcome: {expected_response} Response: {response}""",
    ),
}

In [16]:
# Diagnostic: Test a single metric to ensure Ragas is returning values in 0-1 range
print("=== Ragas Metric Diagnostic ===")
test_question = "When should I charge my EV?"
test_response = "Charge your EV at 2 AM when electricity rates are lowest at $0.12/kWh."
test_expected = "The response should recommend specific hours based on rate and solar forecast."

try:
    result = RESPONSE_METRICS["accuracy"].score(
        llm=ragas_llm,
        question=test_question,
        response=test_response,
        expected_response=test_expected
    )
    raw_value = result.value if hasattr(result, "value") else result
    print(f"Raw result type: {type(raw_value)}")
    print(f"Raw result value: {raw_value}")
    print(f"Float conversion: {float(raw_value)}")
    print(f"Normalized (0-1): {max(0.0, min(1.0, float(raw_value)))}")
    print("✓ Metric diagnostic successful - values are normalized correctly")
except Exception as e:
    print(f"✗ Metric diagnostic failed: {e}")
    print("  Check Ragas metric configuration and OpenAI API access")

=== Ragas Metric Diagnostic ===
Raw result type: <class 'float'>
Raw result value: 0.7
Float conversion: 0.7
Normalized (0-1): 0.7
✓ Metric diagnostic successful - values are normalized correctly


In [17]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate accuracy, relevance, completeness, and usefulness with Ragas."""
    text = (final_response or "").strip()
    if not text or text.lower().startswith("error:"):
        return {
            "accuracy": 0.0, "relevance": 0.0, "completeness": 0.0,
            "usefulness": 0.0, "overall_score": 0.0, "passed": False,
            "feedback": {"general": "The agent did not return a usable final response."},
        }

    scores = {}
    feedback = {}
    for name, metric in RESPONSE_METRICS.items():
        try:
            result = metric.score(
                llm=ragas_llm, question=question, response=text,
                expected_response=expected_response,
            )
            # Extract the numeric value and normalize to 0-1 range
            raw_value = result.value if hasattr(result, "value") else result
            # Ensure the value is a float and normalize if needed
            score_value = float(raw_value)
            # Clamp to 0-1 range
            score_value = max(0.0, min(1.0, score_value))
            scores[name] = round(score_value, 3)
            feedback[name] = result.reason if hasattr(result, "reason") else str(result)
        except Exception as exc:
            scores[name] = 0.0
            feedback[name] = f"Ragas evaluation failed: {exc}"

    overall_score = round(mean(scores.values()), 3)
    return {
        **scores,
        "overall_score": overall_score,
        "passed": overall_score >= 0.70 and all(score >= 0.60 for score in scores.values()),
        "feedback": feedback,
    }

In [13]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate tool appropriateness and completeness with actionable feedback."""
    actual_tools = []
    for message in messages or []:
        data = message.model_dump() if hasattr(message, "model_dump") else message
        if isinstance(data, dict) and data.get("tool_call_id"):
            tool_name = data.get("name")
            if tool_name and tool_name not in actual_tools:
                actual_tools.append(tool_name)

    expected = list(dict.fromkeys(expected_tools or []))
    matched = [tool for tool in expected if tool in actual_tools]
    missing = [tool for tool in expected if tool not in actual_tools]
    unexpected = [tool for tool in actual_tools if tool not in expected]
    recall = len(matched) / len(expected) if expected else 1.0
    precision = len(matched) / len(actual_tools) if actual_tools else (1.0 if not expected else 0.0)

    appropriateness = precision
    completeness = recall
    score = (appropriateness + completeness) / 2
    feedback = []
    if missing:
        feedback.append(f"Missing expected tools: {', '.join(missing)}.")
    if unexpected:
        feedback.append(f"Review whether these additional tools were necessary: {', '.join(unexpected)}.")
    if not actual_tools and expected:
        feedback.append("The agent answered without using any required data tools.")
    if not feedback:
        feedback.append("The agent selected all expected tools and no unrelated tools.")

    return {
        "score": round(score, 3),
        "tool_appropriateness": round(appropriateness, 3),
        "tool_completeness": round(completeness, 3),
        "passed": not missing and appropriateness >= 0.50,
        "expected_tools": expected,
        "actual_tools": actual_tools,
        "matched_tools": matched,
        "missing_tools": missing,
        "unexpected_tools": unexpected,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "feedback": feedback,
    }

In [22]:
def generate_evaluation_report():
    """Generate a comprehensive evaluation report from all test results."""
    evaluations = []
    response_scores = {"accuracy": [], "relevance": [], "completeness": [], "usefulness": []}
    tool_scores = []
    tool_appropriateness_scores = []
    tool_completeness_scores = []
    overall_scores = []
    
    for test_result in test_results:
        test_id = test_result["test_id"]
        question = test_result["question"]
        expected_response = test_result["expected_response"]
        expected_tools = test_result["expected_tools"]
        
        # Extract final response text
        messages = test_result["response"].get("messages", []) if isinstance(test_result["response"], dict) else []
        final_response_text = ""
        if messages:
            last_msg = messages[-1]
            final_response_text = last_msg.content if hasattr(last_msg, "content") else str(last_msg)
        
        # Evaluate response quality
        response_eval = evaluate_response(question, final_response_text, expected_response)
        
        # Evaluate tool usage
        tool_eval = evaluate_tool_usage(messages, expected_tools)
        
        # Calculate overall score
        response_score = max(0.0, min(1.0, response_eval["overall_score"]))
        tool_score = max(0.0, min(1.0, tool_eval["score"]))
        overall_score = (response_score + tool_score) / 2
        
        # Track scores
        for metric in response_scores:
            response_scores[metric].append(max(0.0, min(1.0, response_eval[metric])))
        tool_scores.append(tool_score)
        tool_appropriateness_scores.append(max(0.0, min(1.0, tool_eval["tool_appropriateness"])))
        tool_completeness_scores.append(max(0.0, min(1.0, tool_eval["tool_completeness"])))
        overall_scores.append(overall_score)
        
        # Determine if test passed
        passed = response_eval["passed"] and tool_eval["passed"]
        
        # Create evaluation item
        evaluation_item = {
            "test_id": test_id,
            "passed": passed,
            "overall_score": overall_score,
            "response_evaluation": response_eval,
            "tool_evaluation": tool_eval,
            "error": test_result.get("error", None),
        }
        evaluations.append(evaluation_item)
    
    # Compute aggregates
    tests_run = len(test_results)
    tests_passed = sum(1 for e in evaluations if e["passed"])
    pass_rate = tests_passed / tests_run if tests_run > 0 else 0.0
    
    response_metric_averages = {
        name: mean(scores) if scores else 0.0
        for name, scores in response_scores.items()
    }
    
    average_response_score = mean([e["response_evaluation"]["overall_score"] for e in evaluations]) if evaluations else 0.0
    average_tool_score = mean(tool_scores) if tool_scores else 0.0
    average_tool_appropriateness = mean(tool_appropriateness_scores) if tool_appropriateness_scores else 0.0
    average_tool_completeness = mean(tool_completeness_scores) if tool_completeness_scores else 0.0
    average_overall_score = mean(overall_scores) if overall_scores else 0.0
    
    # Identify strengths
    strengths = []
    if average_response_score >= 0.75:
        strengths.append(f"Response quality is strong ({average_response_score:.0%} average)")
    if response_metric_averages["accuracy"] >= 0.75:
        strengths.append("Accuracy is consistently high across test cases")
    if response_metric_averages["completeness"] >= 0.75:
        strengths.append("Responses are comprehensive and well-structured")
    if average_tool_appropriateness >= 0.75:
        strengths.append("Tool selection is appropriate for the questions asked")
    if pass_rate >= 0.70:
        strengths.append(f"Test pass rate is good ({pass_rate:.0%})")
    
    if not strengths:
        strengths.append("Agent demonstrates competence in basic energy advisory tasks")
    
    # Identify weaknesses
    weaknesses = []
    if pass_rate < 0.70:
        weaknesses.append(f"Only {pass_rate:.0%} of tests pass; review failing cases")
    if average_response_score < 0.70:
        weaknesses.append(f"Average response score is low ({average_response_score:.0%})")
    if response_metric_averages["accuracy"] < 0.70:
        weaknesses.append("Accuracy could be improved; verify calculations and facts")
    if response_metric_averages["completeness"] < 0.70:
        weaknesses.append("Responses are often incomplete; address all aspects of questions")
    if average_tool_completeness < 0.60:
        weaknesses.append("Agent is missing required tools; enhance tool awareness")
    if response_metric_averages["usefulness"] < 0.70:
        weaknesses.append("Recommendations lack actionability; improve specificity")
    
    if not weaknesses:
        weaknesses.append("Minor refinements could further improve recommendation specificity")
    
    # Generate recommendations
    recommendations = []
    if pass_rate < 0.70:
        failing_tests = [e["test_id"] for e in evaluations if not e["passed"]]
        recommendations.append(f"Review failing test cases ({', '.join(failing_tests[:3])}) and refine system prompt")
    if response_metric_averages["accuracy"] < 0.70:
        recommendations.append("Verify energy calculations, pricing assumptions, and data source accuracy in tool responses")
    if average_tool_completeness < 0.60:
        recommendations.append("Audit agent tool awareness; verify all energy-related tools are in the knowledge base")
    if response_metric_averages["relevance"] < 0.75:
        recommendations.append("Improve prompt to stay focused on the user's specific question; reduce tangents")
    if response_metric_averages["usefulness"] < 0.75:
        recommendations.append("Make recommendations more specific and actionable; include time windows, thresholds, and constraints")
    if not recommendations:
        recommendations.append("Continue validation across diverse household profiles and regional conditions")
    
    return {
        "tests_run": tests_run,
        "tests_passed": tests_passed,
        "pass_rate": pass_rate,
        "response_metric_averages": response_metric_averages,
        "average_response_score": average_response_score,
        "average_tool_score": average_tool_score,
        "average_tool_appropriateness": average_tool_appropriateness,
        "average_tool_completeness": average_tool_completeness,
        "average_overall_score": average_overall_score,
        "evaluations": evaluations,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }


In [ ]:
def display_evaluation_report(report=None):
    """Display a readable evaluation report separately from its calculation."""
    if report is None:
        report = generate_evaluation_report()
    
    print("=== ENERGY ADVISOR EVALUATION REPORT ===")
    print(f"Tests passed: {report['tests_passed']}/{report['tests_run']} ({report['pass_rate']:.0%})\n")
    
    print("Response Quality Metrics:")
    for name, score in report["response_metric_averages"].items():
        # Ensure score is normalized to 0-1
        normalized_score = max(0.0, min(1.0, score))
        print(f"  {name.title()}: {normalized_score:.1%}")
    
    print(f"\nAverage response quality: {max(0.0, min(1.0, report['average_response_score'])):.1%}")
    print(f"Average tool score: {max(0.0, min(1.0, report['average_tool_score'])):.1%}")
    print(f"  - Tool appropriateness: {max(0.0, min(1.0, report['average_tool_appropriateness'])):.1%}")
    print(f"  - Tool completeness: {max(0.0, min(1.0, report['average_tool_completeness'])):.1%}")
    print(f"\nAverage overall score: {max(0.0, min(1.0, report['average_overall_score'])):.1%}\n")

    print("Test Results:")
    for item in report["evaluations"]:
        status = "✓ PASS" if item["passed"] else "✗ REVIEW"
        score = max(0.0, min(1.0, item["overall_score"]))
        print(f"{status} {item['test_id']}: {score:.1%}")
        if not item["passed"]:
            if item.get("error"):
                print(f"  ⚠ Error: {item['error']}")
            for metric, note in item["response_evaluation"]["feedback"].items():
                if metric != "general" and note and not note.startswith("Ragas"):
                    print(f"  - {metric.title()}: {note[:60]}...")
            for note in item["tool_evaluation"]["feedback"]:
                print(f"  - Tools: {note}")

    print(f"\n=== STRENGTHS ===")
    for item in report["strengths"]:
        print(f"✓ {item}")

    print(f"\n=== WEAKNESSES ===")
    for item in report["weaknesses"]:
        print(f"✗ {item}")

    print(f"\n=== RECOMMENDATIONS ===")
    for i, item in enumerate(report["recommendations"], 1):
        print(f"{i}. {item}")

# Generate report and display
display_evaluation_report()

=== ENERGY ADVISOR EVALUATION REPORT ===
Tests passed: 5/10 (50%)

Response Quality Metrics:
  Accuracy: 91.0%
  Relevance: 89.0%
  Completeness: 88.0%
  Usefulness: 84.0%

Average response quality: 88.0%
Average tool score: 85.8%
  - Tool appropriateness: 95.0%
  - Tool completeness: 76.7%

Average overall score: 86.9%

Test Results:
✓ PASS ev_charging_1: 98.8%
✓ PASS ev_charging_2: 97.5%
✗ REVIEW thermostat_1: 82.1%
  - Accuracy: The response provides a clear thermostat schedule that align...
  - Relevance: The response provides a detailed thermostat schedule with sp...
  - Completeness: The response provides a detailed thermostat schedule that in...
  - Usefulness: The response provides a clear thermostat schedule that optim...
  - Tools: Missing expected tools: get_weather_forecast, search_energy_tips.
✗ REVIEW thermostat_2: 70.0%
  - Accuracy: The response provides general recommendations for reducing H...
  - Relevance: The response provides general recommendations for reducing H